# Anonymisation benchmark

Runs every available engine over the evaluation dataset and logs the results to MLflow.

**Prerequisites**

1. The API is running: `uv run uvicorn app.main:create_app --factory`
2. `uv sync --group notebook`

Documents are sent over HTTP rather than calling the engines directly, so what is
measured is what the service actually serves — overlap resolution, token limits and all.

In [ ]:
import sys
from pathlib import Path

# Find the project root by walking up until pyproject.toml appears, rather
# than assuming the working directory. `jupyter lab` from the repo root and
# `jupyter execute notebooks/...` give different CWDs, and hardcoding "../"
# silently points every path one level too high - which logs an experiment
# with no runs in it and no error to explain why.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    None,
)
assert ROOT is not None, "could not locate the project root (no pyproject.toml found)"
sys.path.insert(0, str(ROOT))

import pandas as pd

from app.evaluation import runner, tracking
from app.evaluation.metrics import summary_metrics

pd.set_option("display.width", 200)

API_URL = "http://localhost:8000"
DATA_DIR = ROOT / "data"

# MLflow 3 put the classic ./mlruns file store into maintenance mode, so a
# SQLite backend is the local default now. Absolute path, so the notebook and
# `mlflow ui` always agree on which database they mean.
MLFLOW_DB = ROOT / "mlflow.db"
TRACKING_URI = f"sqlite:///{MLFLOW_DB.as_posix()}"
EXPERIMENT = "pii-anonymisation"

print(f"root:     {ROOT}")
print(f"data:     {DATA_DIR}  (exists: {DATA_DIR.exists()})")
print(f"tracking: {TRACKING_URI}")

### Preflight

Fails loudly here rather than producing an empty MLflow experiment later.

In [ ]:
import requests

try:
    health = requests.get(f"{API_URL}/health", timeout=10).json()
except Exception as exc:
    raise SystemExit(
        f"API not reachable at {API_URL}. Start it with:\n"
        "   uv run uvicorn app.main:create_app --factory"
    ) from exc

print("API:      ", health)

required = ["evaluation.jsonl"]
missing = [f for f in required if not (DATA_DIR / f).exists()]
assert not missing, (
    f"missing datasets in {DATA_DIR}: {missing}\n"
    "Build them with: uv run scripts/build_dataset.py"
)
print("datasets: ", required)

## 1. The evaluation dataset

One consolidated file, `data/evaluation.jsonl` — 50 documents, 177 spans,
each record tagged with the subset it came from:

| `source` | Docs | Spans | What it is |
| --- | --- | --- | --- |
| `provided` | 10 | 46 | The supplied set, annotations corrected |
| `belgian` | 20 | 78 | Realistic Belgian documents, all 12 labels |
| `edge` | 20 | 53 | Format variants, distractors, Dutch and French |

Records are `{id, source, text, entities:[{start, end, label}]}`. Spans rather
than masked strings: per-entity scoring needs character offsets, and a masked
string cannot say which of two identical surface forms was caught.


Rebuild either from the CSV sources with `uv run scripts/build_dataset.py`.

In [ ]:
DATASET = DATA_DIR / "evaluation.jsonl"

dataset = runner.load_dataset(DATASET)
subsets = runner.sources(DATASET)

print(f"{len(dataset)} documents, {sum(len(r['entities']) for r in dataset)} gold spans")
print(f"sources: {subsets}")

pd.DataFrame([
    {
        "source": s,
        "docs": len(runner.load_dataset(DATASET, source=s)),
        "spans": sum(len(r["entities"]) for r in runner.load_dataset(DATASET, source=s)),
    }
    for s in subsets
])

In [ ]:
# Label distribution per source. A label with support of 1 gives an F1 of
# either 0 or 1, so it is worth knowing which numbers have no resolution.
from collections import Counter

pd.DataFrame({
    s: Counter(
        e["label"]
        for r in runner.load_dataset(DATASET, source=s)
        for e in r["entities"]
    )
    for s in subsets
}).fillna(0).astype(int).sort_index()

## 2. Which engines are available

The LLM engines only appear if `COHERE_API_KEY` is set in `.env`.

In [ ]:
engines = runner.available_engines(API_URL)
engines

## 3. Run every engine

One HTTP call per document per engine. Each engine becomes one MLflow run,
nested under a parent run for this benchmark, so the compare view lines them up.

A document that errors is scored as a total miss rather than skipped — an engine
that fails on hard inputs should not outscore one that answers them badly.

In [ ]:
# One MLflow run per engine, over the whole evaluation dataset.
# The prompt each LLM engine actually ran is read back from the API and
# logged as an artefact, fingerprinted into a prompt_hash param.
results = runner.run_benchmark(
    dataset_path=DATASET,
    engines=engines,
    api_url=API_URL,
    mode="strict",
    experiment=EXPERIMENT,
    tracking_uri=TRACKING_URI,
    log_errors=True,
)

print(f"logged {len(results)} runs: {list(results)}")

## 4. Headline comparison

Safety first, then quality, then cost.

**`doc_leakage`** is the fraction of documents still containing at least one
unredacted entity. It is the number that decides whether output can be released,
and it is *not* derivable from span recall — at 0.95 recall with five entities per
document, roughly 22% of documents still leak something.

**`micro_f2`** weights recall four times precision. A missed IBAN is a breach; a
false positive is an over-redacted word. F1 treats those as equal, which is wrong here.

In [ ]:
rows = [{"engine": e, **summary_metrics(ev)} for e, ev in results.items()]
pd.DataFrame(rows).round(3).set_index("engine").T

## 5. Per-label recall

Where the real story is. Presidio cannot produce `ORG`, `JOB`, `UNIVERSITY` or
`AMOUNT` at all — they are outside its default entity set — while `IBAN` and `SSN`
are decided by a mod-97 checksum rather than inference. The LLM engines are the
answer to the first group; arithmetic is the answer to the second.

In [ ]:
recall = pd.DataFrame({
    engine: {l: s.recall for l, s in ev.per_label.items() if s.support}
    for engine, ev in results.items()
}).round(2).sort_index()

recall.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.2f}")

In [ ]:
# Precision too, to catch engines buying recall with over-redaction.
precision = pd.DataFrame({
    engine: {l: s.precision for l, s in ev.per_label.items() if s.support}
    for engine, ev in results.items()
}).round(2).sort_index()

precision.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.2f}")

## 6. What leaked

Aggregate numbers say how much; this says what. Reading the actual misses is the
highest-value activity in a classifier project.

In [ ]:
# The artefact logged to MLflow carries the same detail, with the full
# source document for every failing case.
for engine, ev in results.items():
    leaked = [d for d in ev.documents if d.leaked]
    print(f"\n{engine}: {len(leaked)}/{len(ev.documents)} documents leaked")
    for d in leaked[:5]:
        print(f"   {d.doc_id}: {d.text[:70]}")
        for e in d.missed[:3]:
            print(f"        missed {e.label}: {e.text!r}")

In [ ]:
for engine, ev in results.items():
    spurious = [(d, e) for d in ev.documents for e in d.spurious]
    print(f"\n{engine}: {len(spurious)} spurious spans")
    for d, e in spurious[:5]:
        print(f"   {d.doc_id} {e.label}: {e.text!r}")

## 7. Side-by-side output

Sometimes the most informative view is simply the redacted text.

In [ ]:
from app.entities import anonymize

doc_index = 0
record = dataset[doc_index]

print("ORIGINAL\n", record["text"], "\n")
print("GOLD\n", anonymize(record["text"], runner.gold_entities(record)), "\n")

for engine in results:
    import requests
    body = requests.post(
        f"{API_URL}/v1/anonymize",
        json={"text": record["text"], "engine": engine},
        timeout=180,
    ).json()
    print(f"{engine.upper()}\n", body["redacted_text"], "\n")

## 8. All runs in MLflow

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Every run records the engine, dataset, match mode, document count and git SHA as
params, so a run from three weeks ago is still interpretable.

In [ ]:
import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
runs = mlflow.search_runs(experiment_names=[EXPERIMENT])

cols = [
    "tags.mlflow.runName",
    "params.dataset",
    "params.prompt_hash",
    "metrics.document_leakage_rate",
    "metrics.direct_identifier_recall",
    "metrics.micro_f2",
    "metrics.latency_p95_ms",
]
runs[[c for c in cols if c in runs.columns]].head(20)